In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3
import datetime as dt
try:
    import snowflake.connector
except:
    ! pip install snowflake-connector-python
    import snowflake.connector

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2024-03-25 16:31:42.826093


### Functions

In [3]:
# upload to s3
def upload_to_s3(str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client('s3')
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [4]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')
# sub task
str_subtask = os.getcwd().split('/')[6]
print(f'Subtask: {str_subtask}')
# output
str_dirname_output = './output'

Project: 20231010-gen-xii
Task: 12_dark_scoring
Subtask: 04_swap_in_swap_out_plot


### Output directory

In [5]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

### Read query

In [6]:
str_filepath = './sql/query.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('select\n'
 '    ACCOUNTID as bigAccountId,\n'
 '    ORIGINALSCORE as fltDebtorScore,\n'
 '    SCORECARDVERSION\n'
 'from RAW.SOURCE_MONGO.SCORE_SCORERESULT\n'
 "where STAMPCREATION >= '2024-03-14'\n"
 "and SCORECARDVERSION in ('genxi_v2_3','genxii_v2')\n"
 'order by stampcreation desc')


### Write into df

In [7]:
%%time

# connect to snowflake
conn = snowflake.connector.connect(
    user=USERNAME,
    password=PASSWORD,
    account=ACCOUNT,
    warehouse=WAREHOUSE,
    database=DATABASE,
    schema=SCHEMA,
)
# read data
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()
# rename
dict_rename = {
    'BIGACCOUNTID': 'bigAccountId',
    'FLTDEBTORSCORE': 'fltDebtorScore',
    'SCORECARDVERSION': 'strScoreCardVersion',
}
df.rename(columns=dict_rename, inplace=True)

# show
df

<timed exec>:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


CPU times: user 521 ms, sys: 37.4 ms, total: 558 ms
Wall time: 932 ms


,bigAccountId,fltDebtorScore,strScoreCardVersion
0,7704337,0.123931,genxii_v2
1,7704337,0.181348,genxi_v2_3
2,7704336,0.043904,genxii_v2
3,7704336,0.089489,genxi_v2_3
4,7704336,0.043904,genxii_v2
...,...,...,...
74267,7656878,0.158159,genxi_v2_3
74268,7656878,0.126277,genxii_v2
74269,7656878,0.159840,genxi_v2_3
74270,7656878,0.126277,genxii_v2


### Save

In [8]:
%%time

# save
str_filename = 'df_genxi_genxii_scores.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_parquet(str_local_path, compression='gzip')

CPU times: user 79 ms, sys: 7.47 ms, total: 86.4 ms
Wall time: 101 ms


### Upload to s3

In [9]:
%%time

# upload
upload_to_s3(
    str_local_path=str_local_path, 
    str_bucket_key=f'{str_task}/{str_subtask}/{str_filename}', 
    str_bucket_name=str_project,
)

CPU times: user 107 ms, sys: 14.3 ms, total: 122 ms
Wall time: 273 ms


### Clean-up

In [10]:
os.remove(str_local_path)